# Higgs TTS 3 over vLLM-Omni

Higgs has no in-process path. `model_type: higgs_multimodal_qwen3` is unknown to
`transformers`, the HuggingFace repo ships no `auto_map` and no detokenizer weights, so the
only way to run it is an OpenAI compatible `POST /v1/audio/speech` served by vLLM-Omni. This
notebook covers the whole loop: bootstrap the sidecar venv, fetch the weights into `models/`,
start the server, then drive it through the app's own `HiggsEngine` so nothing here is a
second implementation of the protocol.

## Prerequisites

- One CUDA GPU with at least 24 GB. Only 1xH100 80 GB is validated upstream; below 24 GB the
  engine warns and the server will most likely fail to load the 4B model.
- vLLM pins `torch==2.13.0` exactly, which cannot coexist with the main environment, so the
  server lives in its own virtualenv and is driven as a subprocess. The notebook kernel stays
  in the main environment and only speaks HTTP.
- Higgs TTS 3 is released under a research and non-commercial licence with a creator grant
  covering personal audiobooks when Boson AI is credited.

## What this measures

Quality by ear, speed as real time factor, and memory as GPU bytes held by the server
process. The last one is server side by definition: nothing about Higgs allocates in this
kernel.

In [ ]:
import json
import shlex
import subprocess
import sys
import threading
import time
from dataclasses import dataclass
from pathlib import Path

import requests
from IPython.display import Audio, display

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from lib.conf import (
    higgs_defaults, higgs_model_path, higgs_model_repo, vllm_sidecar_port, vllm_venv_dir
)
from lib.inference import TTSConfig
from lib.inference.higgs import HiggsEngine
from lib.modules.text_processing import BatchConfig, SmartSegmenter

print("repo        ", REPO_ROOT)
print("weights dir ", higgs_model_path)
print("venv dir    ", vllm_venv_dir)

## 1. Bootstrap the sidecar venv

Roughly 15 GB of CUDA wheels, so this is never implicit. Run it once, in a terminal, and
watch it. The cell below only prints the commands; copy them out.

In [ ]:
BOOTSTRAP = f"""
uv venv {shlex.quote(vllm_venv_dir)} --python 3.12
uv pip install --python {shlex.quote(vllm_venv_dir + "/bin/python")} \\
    "vllm==0.28.0" "vllm-omni==0.28.0"
{shlex.quote(vllm_venv_dir + "/bin/vllm-omni")} --version
"""

print(BOOTSTRAP)
print("venv present:", Path(vllm_venv_dir, "bin", "vllm-omni").exists())

## 2. Fetch the weights into the repo

Weights go to `models/higgs-tts-3-4b`, never the HuggingFace cache, so one `E2A_MODELS_DIR`
move takes every checkpoint with it. About 9 GB. Set `DOWNLOAD = True` to actually pull.

In [ ]:
DOWNLOAD = False

if DOWNLOAD:
    from huggingface_hub import snapshot_download

    started = time.time()
    path = snapshot_download(repo_id=higgs_model_repo, local_dir=higgs_model_path)
    print(f"downloaded to {path} in {time.time() - started:.0f}s")
else:
    present = Path(higgs_model_path).exists()
    print(f"repo {higgs_model_repo}")
    print(f"local {higgs_model_path} present={present}")
    print("set DOWNLOAD = True to fetch, or run:")
    print(f"  hf download {higgs_model_repo} --local-dir {higgs_model_path}")

## 3. Start the server

Three details are not optional:

- `PYTORCH_CUDA_ALLOC_CONF` must be **popped**. `lib/conf.py` sets `max_split_size_mb:32`
  process wide for Chatterbox, and it fights vLLM's own allocator.
- `VLLM_USE_DEEP_GEMM=0` and `VLLM_MOE_USE_DEEP_GEMM=0`, otherwise the first request can hang
  on kernel autotuning.
- No top level `--tensor-parallel-size` on a multi GPU box. It is broadcast to every stage and
  breaks Higgs' stage 1 (vllm-omni issue 5003). Use `--stage-overrides` or stay on one GPU.

`--gpu-memory-utilization` is the real memory knob for this engine; the app's per job
`--max_vram` ceiling applies to in-process Chatterbox, not to a separate server process.
Sampling (`temperature`, `top_k`) is also server side: vLLM-Omni reads it from the deploy YAML
`default_sampling_params` and ignores it on the request, which is why the engine never sends
it.

In [ ]:
SERVE = f"""
env -u PYTORCH_CUDA_ALLOC_CONF VLLM_USE_DEEP_GEMM=0 VLLM_MOE_USE_DEEP_GEMM=0 \\
  {shlex.quote(vllm_venv_dir + "/bin/vllm-omni")} serve {shlex.quote(higgs_model_path)} \\
    --omni --trust-remote-code \\
    --host 127.0.0.1 --port {vllm_sidecar_port} \\
    --gpu-memory-utilization {higgs_defaults["gpu_memory_utilization"]} \\
    --max-num-seqs {higgs_defaults["max_num_seqs"]}
"""

print(SERVE)
print("first boot of a 4B model takes minutes, watch for 'Application startup complete'")

## 4. Health check

`/v1/models` is the cheapest proof the server is up, and it also tells you the exact model id
to send back, which is not always the directory name.

In [ ]:
SERVER_URL = f"http://127.0.0.1:{vllm_sidecar_port}"


def wait_for_server(url: str, timeout_sec: int = 900) -> dict:
    """Poll /v1/models until the server answers, returning its model list."""
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        try:
            response = requests.get(f"{url}/v1/models", timeout=5)
            if response.ok:
                return response.json()
        except requests.RequestException:
            pass
        time.sleep(3)
    raise TimeoutError(f"no answer from {url} after {timeout_sec}s")


models = wait_for_server(SERVER_URL, timeout_sec=60)
served_id = models["data"][0]["id"]
print("served model:", served_id)

## 5. GPU memory sampling

The kernel allocates nothing, so `torch.cuda.max_memory_allocated` would report zero. What
matters is what the server holds, which `nvidia-smi` reports per process. Note that vLLM
pre-reserves `gpu_memory_utilization` of the card at boot, so the useful number is not the
absolute figure but how it moves between idle and load, and whether a wider batch pushes it.

In [ ]:
def gpu_used_mb() -> int:
    """Total GPU memory in use right now, in MB, across all processes."""
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True
    )
    return max(int(line) for line in out.stdout.split())


class MemorySampler:
    """Poll GPU memory on a thread and keep the peak seen while the block runs."""

    def __init__(self, interval_sec: float = 0.25) -> None:
        self.interval_sec = interval_sec
        self.peak_mb = 0
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def _loop(self) -> None:
        while not self._stop.is_set():
            self.peak_mb = max(self.peak_mb, gpu_used_mb())
            self._stop.wait(self.interval_sec)

    def __enter__(self) -> "MemorySampler":
        self.peak_mb = gpu_used_mb()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc: object) -> None:
        self._stop.set()
        self._thread.join()


idle_mb = gpu_used_mb()
print(f"idle with the server loaded: {idle_mb} MB")

## 6. Build the engine

`transport="client"` means the engine talks to a server somebody else owns, which is exactly
this notebook's situation. The reference wav is base64 encoded into a data URL by
`set_reference_audio`; vLLM-Omni accepts exactly one reference, and a `ref_text` transcript of
it measurably improves cloning fidelity.

In [ ]:
@dataclass
class RunConfig:
    reference_audio: str | None = None     # path to a 5-20s wav, None uses the default voice
    ref_text: str = ""                     # transcript of that wav, improves cloning
    max_new_tokens: int = higgs_defaults["max_new_tokens"]
    max_tokens_per_batch: int = 48         # segmenter budget, matches the pipeline
    sample_rate: int = 24000               # corrected from the first response
    out_dir: Path = REPO_ROOT / "tmp" / "notebook_out" / "higgs"


cfg = RunConfig()
cfg.out_dir.mkdir(parents=True, exist_ok=True)


def build_engine(concurrency: int) -> HiggsEngine:
    """Build a client transport engine pointed at the running server."""
    return HiggsEngine(TTSConfig(
        engine="higgs",
        transport="client",
        server_url=SERVER_URL,
        sample_rate=cfg.sample_rate,
        reference_audio=cfg.reference_audio,
        params={
            "model_name": served_id,
            "ref_text": cfg.ref_text,
            "max_new_tokens": cfg.max_new_tokens,
            "concurrency": concurrency
        }
    ))


engine = build_engine(concurrency=1)
print("engine sample rate", engine.sample_rate, "loaded", engine.is_loaded)

## 7. Segment the text

The app's `SmartSegmenter` is used rather than a notebook local splitter, so what is measured
here is what the pipeline would send.

In [ ]:
SAMPLE_TEXT = """
The lighthouse keeper had not spoken to another person in eleven weeks.
He kept a log, as the service required, and the log was mostly weather.
Wind from the south-west, thirty knots. Rain after midnight. Nothing sighted.

On the twelfth week a boat came, and it came at the wrong hour, riding the swell
without lights. He watched it from the gallery with the lamp turning behind him,
throwing his shadow out over the water in long slow strokes.

The man who climbed the ladder was younger than expected, and soaked through, and
he carried a leather case that he would not put down. He said he had come about
the records.
"""

segmenter = SmartSegmenter(BatchConfig(max_tokens_per_batch=cfg.max_tokens_per_batch))
result = segmenter.segment_text(SAMPLE_TEXT)
segments = [seg.text for seg in result.segments]

print(result.preview())
print("\nsegments:", len(segments))

## 8. One segment, end to end

`TTSResult.audio_data` is raw signed 16 bit PCM at the rate the server reported. Write it
through a wav header before listening, and check that rate: if it is not 24000 the whole
pipeline has to be rerun with `--sample-rate`, or chapter stitching drifts.

In [ ]:
def save_pcm(name: str, pcm: bytes, rate: int) -> Path:
    """Wrap raw 16 bit mono PCM in a wav container under out_dir."""
    import wave

    path = cfg.out_dir / name
    with wave.open(str(path), "wb") as handle:
        handle.setnchannels(1)
        handle.setsampwidth(2)
        handle.setframerate(rate)
        handle.writeframes(pcm)
    return path


with MemorySampler() as sampler:
    single = engine.generate(segments[0])

path = save_pcm("single.wav", single.get_audio_bytes(), single.sample_rate)
print(f"rate={single.sample_rate}Hz duration={single.duration_sec:.2f}s "
      f"wall={single.inference_time_sec:.2f}s rtf={single.real_time_factor:.2f}")
print(f"gpu peak {sampler.peak_mb} MB, {sampler.peak_mb - idle_mb:+d} MB over idle")
display(Audio(str(path)))

## 9. Concurrency sweep

`generate_batch` fans out over a thread pool; the server does the real batching. The point of
the sweep is where throughput stops improving and whether memory climbs with it. Anything
past `--max-num-seqs` queues server side, so expect the curve to flatten there.

In [ ]:
def run_sweep(concurrency: int, texts: list[str]) -> dict:
    """Render every segment at one concurrency level and return the timing row."""
    pool_engine = build_engine(concurrency)
    started = time.time()
    with MemorySampler() as sampler:
        results = pool_engine.generate_batch(texts)
    wall = time.time() - started
    pool_engine.cleanup()
    audio = sum(r.duration_sec for r in results)
    return {
        "concurrency": concurrency,
        "segments": len(results),
        "audio_sec": round(audio, 2),
        "wall_sec": round(wall, 2),
        "rtf": round(wall / max(audio, 1e-6), 3),
        "peak_gpu_mb": sampler.peak_mb,
        "over_idle_mb": sampler.peak_mb - idle_mb
    }


rows = [run_sweep(n, segments) for n in (1, 2, 4)]
(cfg.out_dir / "generation_metadata.json").write_text(json.dumps(rows, indent=2))

for row in rows:
    print(f"concurrency={row['concurrency']:2d} audio={row['audio_sec']:7.2f}s "
          f"wall={row['wall_sec']:7.2f}s rtf={row['rtf']:6.3f} "
          f"peak={row['peak_gpu_mb']}MB ({row['over_idle_mb']:+d} over idle)")

## 10. Listen to the whole passage

Stitched the way the pipeline stitches chunks, with a short gap between segments, so
boundaries are audible at the same spacing the audiobook would have.

In [ ]:
best = build_engine(concurrency=max(r["concurrency"] for r in rows))
results = best.generate_batch(segments)

gap = b"\x00\x00" * int(results[0].sample_rate * 0.12)
stitched = gap.join(r.get_audio_bytes() for r in results)
path = save_pcm("stitched.wav", stitched, results[0].sample_rate)
print(f"{len(results)} segments, {sum(r.duration_sec for r in results):.1f}s")
display(Audio(str(path)))

## 11. Cross lingual cloning

The model card claims 102 languages with 85 under 5 WER/CER, and publishes multilingual clone
benchmarks, but explicit cross lingual transfer, one reference clip speaking a language it
never spoke, is unconfirmed. This cell is how that gets settled: same reference, same engine,
different target language.

In [ ]:
CROSS_LINGUAL = {
    "en": "The keeper made tea, because that was what one did.",
    "fr": "Le gardien prepara du the, parce que c est ce qu on faisait.",
    "de": "Der Waerter kochte Tee, weil man das eben so machte.",
    "es": "El farero preparo te, porque eso era lo que se hacia.",
    "it": "Il guardiano preparo il te, perche era cosi che si faceva.",
    "ja": "灯台守はお茶をいれた。そうするものだったからだ。",
    "zh": "守塔人泡了茶，因为向来如此。"
}

for lang, line in CROSS_LINGUAL.items():
    rendered = engine.generate(line)
    path = save_pcm(f"cross_{lang}.wav", rendered.get_audio_bytes(), rendered.sample_rate)
    print(f"{lang}  {rendered.duration_sec:5.2f}s  rtf={rendered.real_time_factor:5.2f}  {line}")
    display(Audio(str(path)))

## 12. Shut the server down

The server outlives the notebook on purpose, so a second job does not pay the boot cost. Stop
it explicitly when you are done with the card.

```bash
pkill -f "vllm-omni serve"
```

## What to judge

1. **Cut-offs.** Higgs has no upstream 1000 token cap, but `max_new_tokens` is still a
   ceiling. Check the longest segment against its text.
2. **Speaker consistency.** Every segment is an independent request with the same reference.
   Listen across the stitched file for drift in pitch, pace and room tone.
3. **Sample rate.** If section 8 did not report 24000, the pipeline needs `--sample-rate`.
4. **Throughput.** Where the concurrency curve flattens is the useful `--concurrency` value,
   and it is a property of the server's `--max-num-seqs`, not of the client.
5. **Memory.** `over_idle_mb` growing with concurrency means the server is genuinely batching.
   Flat memory with flat throughput means requests are queueing, not batching.
6. **Cross lingual.** Whether the reference speaker survives a language change, and how much
   accent leaks in. This is the open question the plan lists as unconfirmed.